# Mini-TP 7 · Seguridad con SAIF — REST (FastAPI)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por REST (FastAPI)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** abuso, inyección, DoS y *scraping* para robar el modelo.  
**Controles:** autenticación (API key/JWT) · validación de entrada · *rate limiting*.  
**Elementos SAIF:** bases sólidas · automatizar defensas.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add fastapi httpx scikit-learn numpy
```

In [ ]:
import numpy as np, time, hashlib, datetime
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

d = load_breast_cancer(); sc = StandardScaler().fit(d.data)
X = sc.transform(d.data); y = d.target
modelo = LogisticRegression(max_iter=5000).fit(X, y)
N = X.shape[1]
print("Modelo listo. Features:", N)

## 1. Versión VULNERABLE — cualquiera consulta el modelo, sin límites

In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app_v = FastAPI()
@app_v.post("/predict")
def predict_v(payload: dict):
    arr = np.array(payload["features"]).reshape(1, -1)
    return {"pred": int(modelo.predict(arr)[0])}

cli_v = TestClient(app_v)
# sin credenciales, sin validar, sin límite: todo pasa
print("sin auth:", cli_v.post("/predict", json={"features": X[0].tolist()}).status_code)
# y nada frena 1000 consultas seguidas (scraping para robar el modelo)
print("1000 consultas seguidas: permitidas (riesgo de extracción del modelo)")

## 2. Control principal (GUIADO) — auth + validación + rate limiting

In [ ]:
from fastapi import Depends, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, conlist

API_KEY = "demo-key-123"          # en producción: desde variable de entorno
app = FastAPI()

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
def auth(k: str = Depends(api_key_header)):
    if k != API_KEY:
        raise HTTPException(401, "no autorizado")
    return "cliente"

_hits = {}
def rate_limit(cliente: str = Depends(auth)):
    now = time.time(); w = _hits.setdefault(cliente, [])
    w[:] = [t for t in w if now - t < 60]
    if len(w) >= 5:
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return cliente

class Entrada(BaseModel):
    features: conlist(float, min_length=N, max_length=N)   # tipo, tamaño y número exactos

@app.post("/predict")
def predict(x: Entrada, cliente: str = Depends(rate_limit)):
    arr = np.array(x.features).reshape(1, -1)
    return {"prediccion": int(modelo.predict(arr)[0])}

cli = TestClient(app); h = {"X-API-Key": API_KEY}
print("sin key   :", cli.post("/predict", json={"features": X[0].tolist()}).status_code, "(401)")
print("con key   :", cli.post("/predict", json={"features": X[0].tolist()}, headers=h).status_code, "(200)")
print("mala forma:", cli.post("/predict", json={"features": [0.1, 0.2]}, headers=h).status_code, "(422)")
codes = [cli.post("/predict", json={"features": X[0].tolist()}, headers=h).status_code for _ in range(7)]
print("rate limit:", codes, "(429 al superar 5/min)")

## 3. Tu entrega — completa los `# TODO`
Sobre **tu propio modelo**: (a) agrega **autorización por rol con JWT**, y (b) un
**logging de auditoría** (medida de gobernanza).

In [ ]:
import jwt   # uv add pyjwt
JWT_SECRET = "cambiar-en-produccion-con-32-bytes-o-mas"
AUDITORIA = []

def crear_token(usuario, rol, minutos=15):
    payload = {"sub": usuario, "rol": rol,
               "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=minutos)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

# TODO 1: implementa una dependencia requiere_rol() que:
#   - lea el token Bearer (usa fastapi.security.HTTPBearer)
#   - lo verifique con jwt.decode(..., algorithms=["HS256"])
#   - rechace (403) si el rol no está en {"analista","admin"}
# TODO 2: en el endpoint, registra en AUDITORIA cada predicción
#   (ts, usuario, hash de la entrada, predicción).
# TODO 3: adapta 'Entrada' a las features REALES de tu modelo (rangos con Field(ge=, le=)).
print("Token de ejemplo:", crear_token("jaime", "analista")[:30], "...")

## 4. Reflexión
Responde en pocas líneas:
1. ¿Qué amenaza frena el **rate limiting** y por qué se relaciona con el **robo del modelo**?
2. ¿Qué elemento de SAIF cubre tu control principal?
3. ¿Por qué validar la entrada es también una medida de seguridad y no solo de calidad?

*(Escribe tu respuesta aquí.)*